In [1]:
%run ordini.ipynb

(909, 9)
       articolo     brand        fornitore modalita_acquisto categoria    colore stagione  prezzo_listino  costo_unitario
0  AGO-CAM-0001  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     verde     PE23           109.0           40.47
1  AGO-CAM-0002  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           119.0           47.71
2  AGO-CAM-0003  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  fantasia     PE23           129.0           48.24
3  AGO-CAM-0004  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23            79.0           30.54
4  AGO-CAM-0005  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia    bianco     PE23           139.0           52.70
5  AGO-CAM-0006  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           129.0           49.90
6  AGO-CRA-0007  Ago&Filo  Ago&Filo S.r.l.             fermo  cravatta     rosso     PE23            79.0           31.77
7  AGO-CRA-0008

In [2]:
INIZIO, FINE = "2023-03-02", "2026-03-01"

In [4]:
MEDIE = {
    "normale": (4.5, 11.5),
    "picco": (10, 20),
}

In [5]:
SCAGLIONI = {
    1: (0.30, 32.5, 50), 2: (0.30, 32.5, 50),
    3: (0.40, 25, 42.5), 4: (0.50, 25, 42.5),
    5: (0.60, 25, 37.5), 6: (0.60, 25, 37.5),
    7: (0.70, 20, 32.5), 8: (0.70, 20, 32.5),
}

In [6]:
DISPERSIONE = 5

In [7]:
def inizio_saldi_invernali(anno):
    """Primo giorno feriale prima dell'Epifania; se cade di lunedì, si anticipa al sabato."""
    giorno = pd.Timestamp(f"{anno}-01-05")
    if giorno.day_of_week == 6:          # domenica -> sabato 4
        giorno -= pd.Timedelta(days=1)
    elif giorno.day_of_week == 0:        # lunedì -> sabato 3
        giorno -= pd.Timedelta(days=2)
    return giorno

In [8]:
def inizio_saldi_estivi(anno):
    """Primo sabato di luglio."""
    giorno = pd.Timestamp(f"{anno}-07-01")
    return giorno + pd.Timedelta(days=(5 - giorno.day_of_week) % 7)

In [9]:
def periodi_saldo():
    """Elenco (inizio, stagione in saldo) per tutti i saldi del periodo."""
    saldi = []
    for anno in [2023, 2024, 2025]:
        saldi.append((inizio_saldi_estivi(anno), f"PE{anno % 100}"))
        saldi.append((inizio_saldi_invernali(anno + 1), f"AI{anno % 100}"))
    return saldi

In [10]:
def crea_calendario():
    giorni = pd.DataFrame({"data": pd.date_range(INIZIO, FINE)})
    giorni["weekend"] = giorni["data"].dt.day_of_week >= 5
    giorni["periodo"] = "normale"
    giorni["divieto_promo"] = False
    giorni["settimana_saldo"] = 0
    giorni["sconto_saldo"] = 0.0
    giorni["stagione_in_saldo"] = None
 
    mese, gg = giorni["data"].dt.month, giorni["data"].dt.day
    picco = ((mese == 12) & (gg <= 24)) | ((mese == 5) & (gg >= 15)) | (mese == 6)
    giorni.loc[picco, "periodo"] = "picco"
 
    for inizio, stagione in periodi_saldo():
        pre = (giorni["data"] >= inizio - pd.Timedelta(days=30)) & (giorni["data"] < inizio)
        giorni.loc[pre, "divieto_promo"] = True
        for settimana, (sconto, _, _) in SCAGLIONI.items():
            da = inizio + pd.Timedelta(weeks=settimana - 1)
            sett = (giorni["data"] >= da) & (giorni["data"] < da + pd.Timedelta(weeks=1))
            giorni.loc[sett, ["periodo", "settimana_saldo", "sconto_saldo", "stagione_in_saldo"]] = \
                ["saldi", settimana, sconto, stagione]
 
    def media(r):
        if r["periodo"] == "saldi":
            _, fer, wk = SCAGLIONI[r["settimana_saldo"]]
        else:
            fer, wk = MEDIE[r["periodo"]]
        return wk if r["weekend"] else fer
    giorni["media"] = giorni.apply(media, axis=1)
    giorni.loc[giorni["data"].dt.day_of_week == 0, "media"] *= 0.7
 
    settimana = giorni["data"].dt.strftime("%G-%V")
    fattore = {s: rng.lognormal(0, 0.15) for s in settimana.unique()}
    giorni["media"] *= settimana.map(fattore)
 
    p = DISPERSIONE / (DISPERSIONE + giorni["media"])
    giorni["capi"] = rng.negative_binomial(DISPERSIONE, p)
    return giorni

In [12]:
cal = crea_calendario()
print(cal.shape)
print(cal.groupby("periodo")[["media", "capi"]].mean().round(1))
print("\nCapi in dicembre, prima di Natale (deve essere picco):")
print(cal[(cal["data"].dt.month == 12) & (cal["data"].dt.day <= 24)]["periodo"].value_counts())
print("\nInizio saldi:")
print(cal[cal["settimana_saldo"] == 1].groupby("stagione_in_saldo")["data"].min())
print("\nGiorni con divieto di promozioni:", cal["divieto_promo"].sum())
print("\nCapi per anno solare:")
print(cal.groupby(cal["data"].dt.year)["capi"].sum())

(1096, 9)
         media  capi
periodo             
normale    6.4   6.4
picco     12.7  12.7
saldi     29.5  31.4

Capi in dicembre, prima di Natale (deve essere picco):
periodo
picco    72
Name: count, dtype: int64

Inizio saldi:
stagione_in_saldo
AI23   2024-01-05
AI24   2025-01-04
AI25   2026-01-03
PE23   2023-07-01
PE24   2024-07-06
PE25   2025-07-05
Name: data, dtype: datetime64[us]

Giorni con divieto di promozioni: 180

Capi per anno solare:
data
2023    3826
2024    5522
2025    5727
2026    1679
Name: capi, dtype: int64


/var/folders/d_/pq48znn57_g58dhcpdng69km0000gn/T/ipykernel_5180/135431095.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  giorni["weekend"] = giorni["data"].dt.dayofweek >= 5
/var/folders/d_/pq48znn57_g58dhcpdng69km0000gn/T/ipykernel_5180/2183355164.py:4: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  return giorno + pd.Timedelta(days=(5 - giorno.dayofweek) % 7)
/var/folders/d_/pq48znn57_g58dhcpdng69km0000gn/T/ipykernel_5180/2473464577.py:6: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  elif giorno.dayofweek == 0:        # lunedì -> sabato 3
/var/folders/d_/pq48znn57_g58dhcpdng69km0000gn/T/ipykernel_5180/135431095.py:30: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week inste